In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.getcwd())

import torch
from typing import cast
from omegaconf import OmegaConf, DictConfig
from core.data.module import PretrainData
from core.training import create_kde_sampler
from core.training.setup import setup_device
from core.training.pretrain import setup_pretraining, train
from core.model.bobert import BobertForPretraining

In [2]:
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {setup_device()}")
print(f"Working directory: {os.getcwd()}")

config = cast(DictConfig, OmegaConf.load("./config.yaml"))
print(OmegaConf.to_yaml(config))

PyTorch version: 2.9.0+cu126
Using device: gpu
Working directory: /home/jessiez/projects/bobert
data:
  max_seq_len: 2048
  val_split: 0.1
  min_sr: 2.0
  max_sr: 14.0
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward: 1280
  dropout: 0.1
  local_attention_window: 256
components:
  compile_model: true
  compile_mode: default
  activation_checkpointing: false
pretraining:
  db_path: ./data/beatmap_dataset20k/
  batch_size: 8
  num_epochs: 8
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 16
  muon_lr: 0.02
  muon_wd: 0.01
  adam_lr: 0.0002
  adam_wd: 0.05
  adam_betas:
  - 0.9
  - 0.95
  min_lr: 1.0e-06
  cooldown_type: cosine
  warmup_ratio: 0.1
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./experiments
  difficulty_loss_weight: 1.0
  mlm_loss_weight: 1.0
  masking_ratio: 0.25
  mean_span_length: 4
  sampling:
    kde_bandwidth: 0.2
    num_bins: 100
    strength: 0.5
alignment:
  db_path: ./data/beatmap_dataset175k/
  checkpoint_dir: ./experiments
  mode

In [3]:
sampler_fn = lambda stars: create_kde_sampler(
    stars,
    bandwidth=config.pretraining.sampling.kde_bandwidth,
    num_bins=config.pretraining.sampling.get('num_bins', 100),
    strength=config.pretraining.sampling.get('strength', 0.1),
)

datamodule = PretrainData(config, sampler_fn=sampler_fn)
datamodule.setup()

Selected 16648 beatmaps. Processing in chunks of 5000...


Processing Chunks: 100%|██████████| 4/4 [00:10<00:00,  2.74s/it]


Loaded data for 16625 beatmaps.
KDE sampling - Min weight: 0.7528, Max weight: 2.2200
Data split: 14963 training, 1662 validation


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
model = BobertForPretraining.from_config(config, device)

summary = model.bert.get_summary()
print(f"\n--- BERT Encoder Information ---")
print(f"Total Parameters: {summary['trainable_parameters'] / 1e6:.2f}M")
print(f"Model Dimension: {model.bert.d_model}")
print(f"Number of Heads: {model.bert.n_heads}")
print(f"Number of Layers: {model.bert.n_layers}")

/home/jessiez/projects/bobert/.venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Compiling BERT pre-training model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 18.10M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6


In [5]:
module, trainer = setup_pretraining(config, datamodule, model)

print(f"\nPretraining setup complete.")
print(f"Total epochs: {config.pretraining.num_epochs}")
print(f"Training samples: {len(datamodule.train_dataset)}")
print(f"Validation samples: {len(datamodule.val_dataset)}")

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores



Pretraining setup complete.
Total epochs: 8
Training samples: 14963
Validation samples: 1662


In [6]:
train(module, trainer, datamodule)

print("\nBoBERT training completed!")

/home/jessiez/projects/bobert/.venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/jessiez/projects/bobert/experiments/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


Optimizer initialized: 24 Muon params, 40 AdamW params.
Scheduler: WSD with 93 warmup, 93 stable, 750 decay steps.
Cooldown type: cosine, Min LR Ratio: 0.0050
Running warmup pass to initialize RoPE cache to max_seq_len...
Warmup complete. RoPE cache initialized for L=2048 using torch.bfloat16.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

W0428 17:18:26.819000 37716 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1558] [10/2] Not enough SMs to use max_autotune_gemm mode


Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=8` reached.



BoBERT training completed!
